# D7 강화학습 (개념) — 실습 (W14, 2학기 완결편)

> ⚠️ **가장 먼저 — 화면 위 [Drive로 복사]를 누르세요.**
> 지금 보고 있는 것은 원본을 잠깐 띄운 **임시 사본**입니다. 복사하지 않으면 탭을 닫는 순간
> 채운 빈칸과 실행 결과가 **모두 사라집니다.** 복사본은 내 Google Drive에 저장되고,
> 원본은 바뀌지 않으니 마음껏 고쳐도 됩니다.

> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.
> (순수 numpy — GPU·다운로드 불필요, 전부 수 초~수십 초.)

**이 실습이 끝나면**
1. 밴딧으로 **증분 평균**(D1b 리듬)과 **탐험률의 세 운명**(1.019/1.267/1.339/0.750)을 실측한다
2. **Q-learning 갱신 두 걸음을 손계산**(0.5 → 0.2175)으로 완주한다 — "보상이 뒤로 번진다"
3. 격자 세계 800판 학습으로 **최단 경로 7칸**과 정책 화살표를 얻는다
4. **손으로 예언한 시작점 가치 0.5928**을 학습된 Q표가 재현하는 순간을 본다

**7단계 멘탈모델 초점:** 모델 + 활용 (정답 대신 보상으로 학습)

## Part A. 멀티암드 밴딧 — 탐험률의 세 운명 ⭐
슬롯머신 10대, 평균 보상은 모름. 행동별 추정 Q[a]를 **증분 평균**으로 수정하며,
확률 ε로 탐험(무작위)·1−ε로 활용(추정 최고). **ε 네 값의 운명**을 실측합니다.

In [ ]:
import warnings; warnings.filterwarnings('ignore')   # 출력 깔끔하게
import numpy as np                                    # 수치 계산(신경망 불필요!)
import matplotlib.pyplot as plt                       # 시각화

K = 10; STEPS = 1000; RUNS = 200                      # 손잡이 10개, 1000스텝, 200회 평균
def run_bandit(eps, seed):
    rng = np.random.default_rng(seed)                 # 재현성
    true = rng.normal(0, 1, K)                        # 각 행동의 진짜 평균 보상(에이전트는 모름)
    Q = np.zeros(K); N = np.zeros(K); out = np.zeros(STEPS)  # 추정값·선택 횟수
    for t in range(STEPS):
        if rng.random() < eps:
            a = rng.integers(K)                       # 탐험: 무작위 행동
        else:
            a = int(np.___(Q))                        # ✍️ 빈칸: 활용 = 추정 최고의 인덱스
        r = rng.normal(true[a], 1)                    # 보상(노이즈 포함)
        N[a] += 1; Q[a] += (r - Q[a]) / ___           # ✍️ 빈칸: 증분 평균 — 몇 번째 관측으로 나누나?
        out[t] = r
    return out

for eps, col in [(0.0, '#dc2626'), (0.01, '#ca8a04'), (0.1, '#2563eb'), (0.5, '#64748b')]:
    curve = np.mean([run_bandit(eps, s) for s in range(RUNS)], axis=0)  # 200회 평균
    print(f'eps={eps}: 마지막 100스텝 평균 보상 = {curve[-100:].mean():.3f}')
    plt.plot(curve, color=col, label=f'eps={eps}')
plt.xlabel('step'); plt.ylabel('average reward (mean of 200 runs)')  # 축(영어)
plt.title('Bandit: three fates of exploration rate')
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

> **탐험률의 세 운명(+1):** ε=0 → **1.019**(첫인상에 갇힘), ε=0.01 → 1.267(느린 발견), ε=0.1 → **1.339**(적절한 균형·최고), ε=0.5 → **0.750**(절반을 낭비). D1b 학습률 → D4a 쏠림 → D5 온도에 이은 **"적당함" 시리즈의 완결편.** 증분 평균의 "차이만큼 수정"은 D1b 경사하강의 그 리듬.

## Part B. Q-learning 손계산 — 보상이 뒤로 번진다 ⭐
갱신식: `Q[s,a] ← Q[s,a] + α( r + γ·max Q[s′] − Q[s,a] )`, α=0.5, γ=0.95.
격자에서 목표 진입(+1)과 그 한 칸 뒤(−0.04)를 **먼저 종이에서** 완주해 보세요.

In [ ]:
alpha, gamma = 0.5, 0.95                              # 학습률(D1b의 그것)·할인율
q = 0.0                                               # Q[14, 오른쪽] 초기값
q = q + alpha * (1 + gamma * 0.0 - q)                 # ① 목표 진입: r=+1, 목표 뒤엔 미래 없음(0)
print('① 목표 진입 후 Q[14,→] =', q)                 # 0.5
q13 = 0.0                                             # Q[13, 오른쪽] 초기값
q13 = q13 + alpha * (-0.04 + gamma * ___ - q13)       # ✍️ 빈칸: 다음 칸(14)의 최고 가치 = 방금 배운 그 값
print('② 한 칸 뒤    Q[13,→] =', q13)                # 0.2175 — 0.5가 γ를 타고 뒤로 번졌다!
print('   목표 재방문 시:', q + alpha * (1 + 0 - q))  # 0.75 → 0.875 → ... 1.0으로 수렴

> **검산 포인트:** ① 0 + 0.5×(1+0−0) = **0.5**, ② 0 + 0.5×(−0.04 + 0.95×0.5 − 0) = 0.5×0.435 = **0.2175**. 목표의 +1이 **γ 사슬을 타고 한 칸씩 뒤로** — 이 반복이 Q-learning의 전부입니다. γ 곱셈 사슬=D3a·D6 곱셈 쇠퇴의 재회, α와 TD 오차=D1b 학습률·손실의 사촌.

## Part C. 격자 세계 — 800판의 시행착오
4×4, S(0번 칸, 좌상) → GOAL(15번, 우하). 도착 +1, 한 걸음 −0.04(**보상 설계**: "빨리 가라"를 말없이).
동점 Q는 무작위 선택(`amax`) — 초반 전부 0일 때 "위"부터만 골라 탐색이 쏠리는 것을 예방(위생 장치).

In [ ]:
np.random.seed(1)                                     # 재현성
G = 4; nS = G*G; nA = 4; goal = nS - 1                # 4x4, 행동 4(상하좌우), 목표=우하단
def step(s, a):                                       # 환경: 상태+행동 → 다음 상태·보상·종료
    r, cc = divmod(s, G)
    if a == 0: r = max(r-1, 0)                        # up
    elif a == 1: r = min(r+1, G-1)                    # down
    elif a == 2: cc = max(cc-1, 0)                    # left
    else: cc = min(cc+1, G-1)                         # right
    s2 = r*G + cc
    return s2, (1.0 if s2 == goal else -0.04), (s2 == goal)
def amax(qv):                                         # 동점은 무작위(쏠림 방지)
    return np.random.choice(np.flatnonzero(qv == qv.max()))

Q = np.zeros((nS, nA)); alpha = 0.5; gamma = 0.95; eps = 0.2
for ep in range(800):                                 # 800 에피소드 시행착오
    s = 0
    for _ in range(60):
        a = np.random.randint(nA) if np.random.rand() < ___ else amax(Q[s])  # ✍️ 빈칸: 탐험 확률
        s2, r, done = step(s, a)
        Q[s, a] += alpha * (r + gamma * np.___(Q[s2]) - Q[s, a])  # ✍️ 빈칸: 다음 상태의 최고 가치
        s = s2
        if done: break

s = 0; path = [0]                                     # 학습된 정책으로 길 찾기(탐욕)
for _ in range(20):
    a = amax(Q[s]); s, _, done = step(s, a); path.append(s)
    if done: break
print('학습된 경로:', path, '| 칸 수:', len(path), '(최적 7)')

fig, ax = plt.subplots(figsize=(3.6, 3.6))            # 정책 화살표 지도
ax.set_xlim(-.5, G-.5); ax.set_ylim(-.5, G-.5); ax.invert_yaxis()
for rr in range(G):
    for cc in range(G):
        st = rr*G + cc
        if st == goal:
            ax.text(cc, rr, 'GOAL', ha='center', va='center', color='#16a34a', fontweight='bold', fontsize=9)
        else:
            a = int(np.argmax(Q[st])); dx, dy = {0: (0, -.3), 1: (0, .3), 2: (-.3, 0), 3: (.3, 0)}[a]
            ax.arrow(cc, rr, dx, dy, head_width=0.12, color='#2563eb')
ax.text(0, 0.28, 'S', ha='center', color='#dc2626', fontweight='bold', fontsize=9)
ax.set_xticks(range(G)); ax.set_yticks(range(G)); ax.grid(True, alpha=0.3)
ax.set_title('Q-learning: learned policy (S -> GOAL)')  # 제목(영어)
plt.tight_layout(); plt.show()

> **7칸 — 최단 경로.** 정답 라벨 없이, 목표 +1과 걸음 −0.04만으로 길이 생겼습니다. 화살표 지도의 모든 칸이 GOAL 쪽을 가리키는 것도 확인하세요(경로 밖 칸들도 배웠다 — 탐험 덕).

## Part D. 예언과 실측 — 0.5928의 순간 ⭐
수렴하면 시작점의 가치는 얼마여야 할까요? **Q표의 값을 열어 보기 전에**, 최적 경로 6걸음을 거꾸로 손으로 쌓아 예언합니다: V = −0.04 + 0.95×V(다음 칸).

In [ ]:
v = 1.0                                               # 목표 앞 칸의 수렴 가치(손계산 ①)
chain = [v]
for _ in range(5):                                    # 최적 경로 6걸음을 거꾸로
    v = -0.04 + gamma * ___                           # ✍️ 빈칸: 한 칸 앞의 가치에 할인·걸음값 적용
    chain.append(v)
print('예언 사슬(목표 앞→시작):', [round(x, 4) for x in chain])
print('예언한 시작점 가치:', round(v, 4))             # 0.5928
print('실측 Q[시작].max()  =', round(Q[0].max(), 4))  # 일치하는가?
print('실측 Q[14].max()   =', round(Q[14].max(), 4), '| Q[13].max() =', round(Q[13].max(), 4))

V = Q.max(axis=1).reshape(G, G)                       # 상태 가치 지도 V(s)=max Q
fig, ax = plt.subplots(figsize=(4.4, 3.6))
im = ax.imshow(V, cmap='YlGn')                        # 진할수록 가치 높음
for rr in range(G):
    for cc in range(G):
        ax.text(cc, rr, f'{V[rr, cc]:.2f}', ha='center', va='center', fontsize=9,
                color='black' if V[rr, cc] < 0.75 else 'white')
ax.set_title('Learned state value V(s) = max Q')      # 제목(영어)
ax.set_xticks(range(G)); ax.set_yticks(range(G))
fig.colorbar(im, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()

> **예언 0.5928 = 실측 Q[시작].max() 0.5928 — 소수 넷째 자리까지 일치!** (Q[14]=1.0, Q[13]=0.91도 사슬 그대로.) V맵에서 가치가 GOAL에서 S 쪽으로 **등고선처럼 번진** 모습 — "보상이 뒤로 번진다"의 완성 사진입니다. 부트스트랩(추정으로 추정 다듬기)이 정말 수렴한다는 실증.

## 강화학습은 어디에 — 그리고 D5와의 재회
- **게임·제어:** AlphaGo·아타리·로봇(시뮬레이터 = 싼 시행착오)
- **추천·운영:** 장기 보상(유지·전환)을 키우는 선택
- **RLHF:** D5의 LLM — 좋은 답은 하나가 아니라 **정답 라벨 불가**, 하지만 **선호 비교는 가능** → 사람 선호를 보상 삼아 답변 정책을 다듬기(오늘의 챗봇이 오늘처럼 답하는 비결의 한 축)

> 난점: **보상 설계**(−0.04에도 의도가 있었듯 — 잘못 설계하면 편법=보상 해킹) + **시행착오 비용**.

## 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "Q[13,→] = 0.2175가 나오는 계산을 내가 완주해 볼 테니 채점해 줘."
- "예언 사슬 1.0→0.91→…→0.5928을 유도해 볼게 (V=−0.04+0.95V)."
- "ε=0이 1.019에 갇히는 이유를 '첫인상'으로 설명해 볼게 — 반례 상황을 들어 줘."
- "−0.04를 0으로 바꾸면 무슨 일이 생길지 예측해 볼게 — 실행으로 검증할게."

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력은 실행으로 검증

## 정리 & 자가 점검 — 2학기 대장정 완주 🎉

**오늘 한 일 3줄**
1. 밴딧으로 탐험률의 세 운명(1.019/1.267/**1.339**/0.750)을 실측했다 — "적당함" 시리즈 완결
2. Q-learning 두 걸음을 손계산(0.5→0.2175)하고, 800판 학습으로 **최단 경로 7칸**을 얻었다
3. 손으로 예언한 **0.5928**을 학습된 Q표가 재현하는 순간을 봤다 — 보상이 뒤로 번진다

**스스로 점검**
- [ ] 증분 평균과 D1b 경사하강의 공통 리듬("차이만큼 수정")을 안다
- [ ] ε 네 값의 순위와 이유(갇힘/균형/낭비)를 설명할 수 있다
- [ ] 0.2175의 유도 과정과 "뒤로 번짐"의 의미를 안다
- [ ] RLHF가 지도학습이 아니라 강화학습인 이유를 안다

**🔹심화 (선택)**
- **걸음값 실험:** −0.04를 0으로 바꾸면? (헤매도 안 아픔 → 경로가 늘어지는지 실측) / −0.5로 키우면?
- **γ 실험:** gamma=0.5로 낮추면 예언 사슬과 V맵이 어떻게 변할까요? 손계산 후 실측 대조.
- **ε 스케줄:** 초반엔 크게, 후반엔 작게(ε 감쇠) — 세 운명의 좋은 점만 합치기.
- **UCB 맛보기:** "불확실하면 보너스" — ε-탐욕과 비교(심화).

> 🎉 **2학기 딥러닝 완주!** 텐서(D1) → CNN(D2) → 시퀀스(D3) → 트랜스포머(D4) → LLM(D5) → 생성(D6) → **행동(D7)**. 맞히고, 만들고, 행동하는 세 동사를 모두 지났습니다 — 다음은 종합 프로젝트.